# 🔬 Disease Detection System — EDA & Baseline Model Development

This notebook serves as the initial exploratory data analysis and prototyping environment for the **EfficientNetB4** skin lesion detector. 

### Objectives:
1. Verify the local directory layout of the raw dataset.
2. Visualize class distributions and assess severe class imbalances.
3. Compare original images with Contrast Limited Adaptive Histogram Equalization (CLAHE) enhanced versions.
4. Construct the compilation structure of the baseline transfer learning model.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. Load Dataset Directories & Check Class Distribution

In [ ]:
# Define raw training data directory
raw_train_path = Path("../data/raw/isic/Train")

if raw_train_path.exists():
    class_dirs = sorted([d for d in raw_train_path.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_dirs]
    print(f"Detected {len(class_names)} classes: {class_names}")
    
    class_counts = {}
    for c_dir in class_dirs:
        count = len(list(c_dir.glob("*.jpg")) + list(c_dir.glob("*.jpeg")) + list(c_dir.glob("*.png")))
        class_counts[c_dir.name] = count
        
    print("\nImage counts per class:")
    for cls, cnt in class_counts.items():
        print(f"  - {cls}: {cnt} images")
else:
    print("⚠️ Raw dataset path not found. Please ensure data is loaded at data/raw/isic/")
    class_counts = {}
    class_names = []

## 2. Visualize Class Imbalances

In [ ]:
if class_counts:
    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")
    sns.barplot(
        x=list(class_counts.values()),
        y=list(class_counts.keys()),
        palette="viridis"
    )
    plt.title("ISIC Skin Lesion Dataset Class Balance")
    plt.xlabel("Number of Images")
    plt.ylabel("Diagnosis Category")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping plot: No dataset counts loaded.")

## 3. Verify CLAHE Preprocessing

In [ ]:
def apply_clahe(img_path):
    img_bgr = cv2.imread(str(img_path))
    # Convert to LAB space to apply CLAHE to the L channel
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)
    
    lab_enhanced = cv2.merge((l_enhanced, a, b))
    enhanced_bgr = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)
    
    # Convert back to RGB for matplotlib representation
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)

# Locate sample image
sample_image = None
if raw_train_path.exists():
    for c_dir in class_dirs:
        img_list = list(c_dir.glob("*.jpg")) + list(c_dir.glob("*.png"))
        if img_list:
            sample_image = img_list[0]
            break

if sample_image:
    raw, enhanced = apply_clahe(sample_image)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(raw)
    axes[0].set_title("Original Scan")
    axes[0].axis("off")
    
    axes[1].imshow(enhanced)
    axes[1].set_title("CLAHE Preprocessed Scan")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("No sample image found to test CLAHE.")

## 4. Instantiate Baseline Model Architecture

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, Model

num_classes = len(class_names) if class_names else 9

# Build EfficientNetB4 transfer base
base_model = EfficientNetB4(
    include_top=False,
    weights=None,  # Set to None for baseline validation compilation
    input_shape=(380, 380, 3)
)
base_model.trainable = False

# Custom classification layers
inputs = layers.Input(shape=(380, 380, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(512, activation="swish")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("Baseline Model Compiled Successfully!")
print(f"Total Parameters: {model.count_params():,}")